In [1]:
import pandas as pd

df = pd.read_csv("data/raw/master_dataset_real.csv")


In [2]:
df = df[[
    'latitude',
    'longitude',
    'temperature_2m',
    'relative_humidity_2m',
    'wind_speed_10m',
    'soil_moisture_0_to_7cm',
    'acq_date',
    'fire_detected'
]]


In [3]:
df['acq_date'] = pd.to_datetime(df['acq_date'])

df['month'] = df['acq_date'].dt.month
df['day_of_year'] = df['acq_date'].dt.dayofyear

df = df.drop(columns=['acq_date'])


In [4]:
print("FINAL shape:", df.shape)

print(df['fire_detected'].value_counts())
print(df['fire_detected'].value_counts(normalize=True))


FINAL shape: (4976, 9)
fire_detected
1    2488
0    2488
Name: count, dtype: int64
fire_detected
1    0.5
0    0.5
Name: proportion, dtype: float64


In [5]:
df.describe()


,latitude,longitude,temperature_2m,relative_humidity_2m,wind_speed_10m,soil_moisture_0_to_7cm,fire_detected,month,day_of_year
count,4976.000000,4976.000000,4976.000000,4976.000000,4976.000000,0.0,4976.00000,4976.0,4976.000000
mean,22.105057,85.282176,15.281572,71.841439,8.044634,NaN,0.50000,1.0,27.884445
std,6.619805,9.060123,11.751409,13.809515,7.235490,NaN,0.50005,0.0,0.319722
min,8.001721,68.005590,-34.600000,14.000000,0.400000,NaN,0.00000,1.0,27.000000
25%,17.938436,77.198159,13.600000,65.000000,3.400000,NaN,0.00000,1.0,28.000000
50%,21.990735,86.095305,19.100000,74.000000,5.100000,NaN,0.50000,1.0,28.000000
75%,26.434583,94.685296,22.100000,81.000000,9.500000,NaN,1.00000,1.0,28.000000
max,36.965112,96.998370,28.200000,100.000000,50.400000,NaN,1.00000,1.0,28.000000


In [6]:
df['dryness_index'] = (
    df['temperature_2m'] / (df['relative_humidity_2m'] + 1)
)


In [7]:
df = df.drop(columns=['soil_moisture_0_to_7cm'])
# Soil moisture level was not provided consistently across all records.

In [8]:
FEATURES = [
    'latitude',
    'longitude',
    'temperature_2m',
    'relative_humidity_2m',
    'wind_speed_10m',
    'month',
    'day_of_year'
]

X = df[FEATURES]
y = df['fire_detected']

print(X.shape, y.shape)


(4976, 7) (4976,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keeps 50/50 balance
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


Train: (3980, 7)
Test : (996, 7)


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


,n_estimators,300
,criterion,'gini'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))


              precision    recall  f1-score   support

           0       0.90      0.87      0.89       498
           1       0.88      0.90      0.89       498

    accuracy                           0.89       996
   macro avg       0.89      0.89      0.89       996
weighted avg       0.89      0.89      0.89       996

ROC AUC: 0.9609401461266753


In [12]:
import pandas as pd

importance = pd.Series(
    rf.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print(importance)


longitude               0.244404
temperature_2m          0.211515
wind_speed_10m          0.192166
latitude                0.167733
day_of_year             0.123872
relative_humidity_2m    0.060310
month                   0.000000
dtype: float64


In [13]:
import pandas as pd

fires_df = pd.read_csv("data/raw/fire_archive_SV-C2_710630.csv")

print(fires_df.shape)
fires_df.head()


(536656, 15)


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,25.32538,91.59396,342.91,0.74,0.76,2025-01-01,601,N,VIIRS,n,2,292.54,9.84,D,0
1,27.20211,96.98654,343.67,0.32,0.55,2025-01-01,602,N,VIIRS,n,2,292.10,3.17,D,0
2,26.93549,94.04391,346.57,0.49,0.65,2025-01-01,602,N,VIIRS,n,2,293.16,8.22,D,0
3,9.96524,76.39141,328.81,0.34,0.56,2025-01-01,738,N,VIIRS,n,2,301.74,1.99,D,0
4,10.69838,76.53059,338.62,0.33,0.55,2025-01-01,738,N,VIIRS,n,2,301.74,3.62,D,0


In [14]:
import pandas as pd

df_fire = pd.read_csv("data/raw/fire_archive_SV-C2_710630.csv")

# Keep relevant columns
df_fire = df_fire[
    ['latitude', 'longitude', 'brightness', 'bright_t31',
     'frp', 'confidence', 'daynight', 'acq_date']
]

# Label fires
df_fire['fire_detected'] = 1

# Convert date
df_fire['acq_date'] = pd.to_datetime(df_fire['acq_date'])

df_fire['month'] = df_fire['acq_date'].dt.month
df_fire['day_of_year'] = df_fire['acq_date'].dt.dayofyear


In [15]:
df_fire['daynight'] = df_fire['daynight'].map({'D': 1, 'N': 0})


In [16]:
import numpy as np

safe_points = []

for _, row in df_fire.iterrows():
    safe_points.append({
        'latitude': np.random.uniform(8, 37),
        'longitude': np.random.uniform(68, 97),
        'brightness': 0.0,
        'bright_t31': 0.0,
        'frp': 0.0,
        'confidence': 0,
        'daynight': row['daynight'],
        'acq_date': row['acq_date'],
        'fire_detected': 0,
        'month': row['month'],
        'day_of_year': row['day_of_year']
    })

df_safe = pd.DataFrame(safe_points)

df = pd.concat([df_fire, df_safe], ignore_index=True)


In [17]:
FEATURES = [
    'latitude',
    'longitude',
    'brightness',
    'bright_t31',
    'frp',
    'confidence',
    'daynight',
    'month',
    'day_of_year'
]

X = df[FEATURES]
y = df['fire_detected']


In [18]:
train_df = df[df['day_of_year'] <= 300]
test_df  = df[df['day_of_year'] > 300]

X_train = train_df[FEATURES]
y_train = train_df['fire_detected']

X_test = test_df[FEATURES]
y_test = test_df['fire_detected']


In [19]:
# Map FIRMS confidence to numeric
confidence_map = {
    'l': 0,
    'n': 1,
    'h': 2
}

df['confidence'] = df['confidence'].map(confidence_map)


In [20]:
# Convert to numeric if already numeric, else mapped values stay
df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')

# Fill any remaining NaN with lowest confidence
df['confidence'] = df['confidence'].fillna(0)


In [21]:
print(df[FEATURES].dtypes)
print(df[FEATURES].isna().sum())


latitude       float64
longitude      float64
brightness     float64
bright_t31     float64
frp            float64
confidence     float64
daynight         int64
month            int64
day_of_year      int64
dtype: object
latitude       0
longitude      0
brightness     0
bright_t31     0
frp            0
confidence     0
daynight       0
month          0
day_of_year    0
dtype: int64


In [22]:
X = df[FEATURES]
y = df['fire_detected']

train_df = df[df['day_of_year'] <= 300]
test_df  = df[df['day_of_year'] > 300]

X_train = train_df[FEATURES]
y_train = train_df['fire_detected']

X_test = test_df[FEATURES]
y_test = test_df['fire_detected']


In [23]:
rf.fit(X_train, y_train)


,n_estimators,300
,criterion,'gini'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1636
           1       1.00      1.00      1.00      1636

    accuracy                           1.00      3272
   macro avg       1.00      1.00      1.00      3272
weighted avg       1.00      1.00      1.00      3272

ROC-AUC: 1.0


In [25]:
LEAKY_FEATURES = [
    'brightness',
    'bright_t31',
    'frp'
]

df = df.drop(columns=LEAKY_FEATURES)


In [26]:
FEATURES = [
    'latitude',
    'longitude',
    'confidence',   # optional
    'daynight',     # optional
    'month',
    'day_of_year'
]


In [27]:
FEATURES = [
    'latitude',
    'longitude',
    'month',
    'day_of_year'
]

X = df[FEATURES]
y = df['fire_detected']


In [28]:
train_df = df[df['day_of_year'] <= 300]
test_df  = df[df['day_of_year'] > 300]

X_train = train_df[FEATURES]
y_train = train_df['fire_detected']

X_test = test_df[FEATURES]
y_test = test_df['fire_detected']


In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


              precision    recall  f1-score   support

           0       0.88      0.83      0.86      1636
           1       0.84      0.89      0.87      1636

    accuracy                           0.86      3272
   macro avg       0.86      0.86      0.86      3272
weighted avg       0.86      0.86      0.86      3272

ROC-AUC: 0.9272115482332124
